# `kdbox` Interactive Walkthrough

This notebook demonstrates the complete `kdbox` workflow on a synthetic multivariate telemetry dataset:

1. **Data Preparation (`from_dataframe`, `align_time_series`):** Resampling ragged timestamps, trimming long-tail duration outliers, bridging corrupted spikes synchronously across all variables, and applying per-column interpolation.
2. **Univariate Exploration (`KDBoxWidget`):** Spatial queries (*Brush* with *Strict* mode, *RNL*, *Angular*, *Pin*), *Heuristic* vs. *Curve Mean-Shift* representative lines, and two-way programmatic control.
3. **Coordinated Multivariate Exploration (`MultivariateKDBox`):** Cross-filtering across three linked variables, interpreting shaded query time bands in the Matplotlib overlay, and filtering high-cardinality categorical attributes.
4. **Data Export:** Extracting filtered entity indices, aligned arrays, and source DataFrame slices directly back into Python.

In [1]:
import json
import numpy as np
import pandas as pd

import kdbox
from kdbox import KDBoxWidget, MultivariateKDBox, align_time_series, from_dataframe

print(f"kdbox version: {kdbox.__version__}")

kdbox version: 0.1.0


In [2]:
def generate_synthetic_telemetry(seed: int = 42) -> pd.DataFrame:
    """Generate a ragged long-format multivariate telemetry DataFrame.

    Contains 3 operational regimes + injected anomalies:
      - Normal (450 runs):        Steady ~45-51 °C, ~5.2 bar, valve 25% -> 50%.
      - Thermal Surge (200 runs): Mid-run heat spike (~88 °C), correlated pressure
                                  drop (~2.6 bar), emergency valve jump to 100%
                                  (skewed toward Plant North & UNIT-001..006).
      - Cooling Drift (200 runs): Steady downward temp slope (~62 -> 32 °C),
                                  rising pressure (~4.6 -> 6.0 bar), valve 45% -> 20%.
      - Stalled Runs (25 runs):   Anomalously long durations (900-1400 s).
      - Sensor Glitches (~3%):    Isolated 999.9 °C spikes in `temperature`.
    """
    rng = np.random.default_rng(seed)

    n_normal, n_overheat, n_drift, n_stalled = 450, 200, 200, 25
    n_total = n_normal + n_overheat + n_drift + n_stalled

    plants = ["Plant North", "Plant South", "Plant East", "Plant West"]
    # 30 unique units (>12 threshold) to demonstrate the live Attribute Search box.
    unit_ids = [f"UNIT-{i:03d}" for i in range(1, 31)]
    base_ts = pd.Timestamp("2026-01-15 08:00:00", tz="UTC")

    frames = []
    for i in range(n_total):
        run_id = f"RUN-{i + 1:04d}"
        plant = plants[i % len(plants)]
        unit = unit_ids[i % len(unit_ids)]

        if i < n_normal:
            regime = "Normal"
            duration = rng.uniform(120.0, 220.0)
        elif i < n_normal + n_overheat:
            regime = "Thermal Surge"
            duration = rng.uniform(130.0, 210.0)
            plant = rng.choice(["Plant North", "Plant East"], p=[0.75, 0.25])
            unit = rng.choice(unit_ids[:6])
        elif i < n_normal + n_overheat + n_drift:
            regime = "Cooling Drift"
            duration = rng.uniform(140.0, 240.0)
        else:
            regime = "Stalled Run (Outlier)"
            duration = rng.uniform(900.0, 1400.0)

        n_pts = int(rng.integers(35, 75))
        rel_t = np.sort(rng.uniform(0.0, duration, size=n_pts))
        rel_t[0], rel_t[-1] = 0.0, duration
        tau = rel_t / duration

        if regime == "Normal":
            temp = 45.0 + 6.0 * np.sin(np.pi * tau) + rng.normal(0.0, 1.2, size=n_pts)
            pressure = 5.2 + 0.3 * np.cos(np.pi * tau) + rng.normal(0.0, 0.12, size=n_pts)
            valve = np.where(tau < 0.15, 25.0, 50.0)
        elif regime == "Thermal Surge":
            surge = 38.0 * np.exp(-0.5 * ((tau - 0.45) / 0.10) ** 2)
            temp = 50.0 + surge + rng.normal(0.0, 1.5, size=n_pts)
            pressure = 5.0 - 2.4 * np.exp(-0.5 * ((tau - 0.48) / 0.11) ** 2) + rng.normal(0.0, 0.12, size=n_pts)
            valve = np.where((tau >= 0.35) & (tau <= 0.65), 100.0, 50.0)
        elif regime == "Cooling Drift":
            temp = 62.0 - 30.0 * tau + rng.normal(0.0, 1.3, size=n_pts)
            pressure = 4.6 + 1.4 * tau + rng.normal(0.0, 0.12, size=n_pts)
            valve = np.where(tau > 0.50, 20.0, 45.0)
        else:
            temp = 44.0 + rng.normal(0.0, 1.0, size=n_pts)
            pressure = 5.1 + rng.normal(0.0, 0.10, size=n_pts)
            valve = np.full(n_pts, 50.0)

        # Inject isolated corrupted sensor spikes (~3% of runs).
        if i % 35 == 0 and regime != "Stalled Run (Outlier)":
            temp[n_pts // 2] = 999.9

        start_offset = pd.Timedelta(minutes=int(i * 3))
        timestamps = base_ts + start_offset + pd.to_timedelta(rel_t, unit="s")

        frames.append(pd.DataFrame({
            "run_id": run_id,
            "timestamp": timestamps,
            "temperature": temp,
            "pressure": pressure,
            "valve_pct": valve,
            "plant": plant,
            "unit_id": unit,
            "true_regime": regime,
        }))

    return pd.concat(frames, ignore_index=True)


df_raw = generate_synthetic_telemetry()
print(f"Raw DataFrame shape: {df_raw.shape} ({df_raw['run_id'].nunique()} runs)")
print(f"Raw temperature range: [{df_raw['temperature'].min():.1f}, {df_raw['temperature'].max():.1f}] °C")
df_raw.head()

Raw DataFrame shape: (47622, 8) (875 runs)
Raw temperature range: [27.8, 999.9] °C


,run_id,timestamp,temperature,pressure,valve_pct,plant,unit_id,true_regime
0,RUN-0001,2026-01-15 08:00:00.000000000+00:00,44.598138,5.526400,25.0,Plant North,UNIT-001,Normal
1,RUN-0001,2026-01-15 08:00:08.646670843+00:00,46.018382,5.660266,25.0,Plant North,UNIT-001,Normal
2,RUN-0001,2026-01-15 08:00:12.597245869+00:00,46.898351,5.594204,25.0,Plant North,UNIT-001,Normal
3,RUN-0001,2026-01-15 08:00:18.590194550+00:00,47.602887,5.529789,25.0,Plant North,UNIT-001,Normal
4,RUN-0001,2026-01-15 08:00:22.607733140+00:00,48.064577,5.656386,25.0,Plant North,UNIT-001,Normal


## 1. Preprocessing with `from_dataframe` & `align_time_series`

Our raw table has three real-world quirks that would distort a naive plot:
1. **Ragged, irregular timestamps** across 875 runs.
2. **25 stalled runs lasting > 900 s** (while normal runs finish in 120--240 s).
3. **25 corrupted `999.9°C` sensor glitches** in `temperature`.

`from_dataframe` fixes all three in one synchronized pass:
* `max_duration_quantile=0.97` automatically trims the long-tail stalled runs across **all** variables.
* `value_range={"temperature": (0, 120)}` with `value_out_of_range="drop_points"` drops glitch rows across all variables simultaneously so linear interpolation bridges them cleanly.
* Per-column `kind` uses `"linear"` for continuous sensors (`temperature`, `pressure`) and `"previous"` (step hold) for the discrete `valve_pct` setting.

In [3]:
prep = from_dataframe(
    df_raw,
    id_col="run_id",
    time_col="timestamp",
    value_cols=["temperature", "pressure", "valve_pct"],
    attr_cols=["plant", "unit_id", "true_regime"],
    n_steps=100,
    align="start",
    time_unit="s",
    max_duration_quantile=0.97,
    value_range={"temperature": (0.0, 120.0)},
    value_out_of_range="drop_points",
    kind={
        "temperature": "linear",
        "pressure": "linear",
        "valve_pct": "previous",
    },
)

print(f"Prepared runs : {len(prep.ids)} (trimmed stalled outliers)")
print(f"Time shape    : {prep.time_data.shape}, range=[{prep.time_data.min():.1f}, {prep.time_data.max():.1f}] s")
print(f"Temp range    : [{prep.value_data['temperature'].min():.1f}, {prep.value_data['temperature'].max():.1f}] °C")
print(f"Pressure range: [{prep.value_data['pressure'].min():.2f}, {prep.value_data['pressure'].max():.2f}] bar")
print(f"Attributes    : {list(prep.attributes.keys())}")

Prepared runs : 848 (trimmed stalled outliers)
Time shape    : (848, 100), range=[0.0, 238.7] s
Temp range    : [27.8, 93.2] °C
Pressure range: [2.29, 6.36] bar
Attributes    : ['plant', 'unit_id', 'true_regime']


In [4]:
# Compare alignment modes on an existing 2D time matrix without re-running from_dataframe.
for mode in ("start", "end", "center", "normalize"):
    t_aligned = align_time_series(prep.time_data, align=mode)
    print(f"align={mode:<9s} -> global range [{t_aligned.min():6.1f}, {t_aligned.max():6.1f}]")

# You can also slice a specific aligned time window directly in from_dataframe
# (e.g., focus exclusively on the first 150 seconds of every run):
prep_first_150s = from_dataframe(
    df_raw,
    id_col="run_id",
    time_col="timestamp",
    value_cols="temperature",
    n_steps=100,
    align="start",
    max_duration_quantile=0.97,
    time_range=(0.0, 150.0),
    time_out_of_range="slice",
    value_range=(0.0, 120.0),
    value_out_of_range="drop_points",
)
print(
    f"\nSliced first 150s -> shape={prep_first_150s.time_data.shape}, "
    f"range=[{prep_first_150s.time_data.min():.1f}, {prep_first_150s.time_data.max():.1f}] s"
)

align=start     -> global range [   0.0,  238.7]
align=end       -> global range [-238.7,    0.0]
align=center    -> global range [-119.3,  119.3]
align=normalize -> global range [   0.0,    1.0]

Sliced first 150s -> shape=(848, 100), range=[0.0, 150.0] s


## 2. Univariate Exploration (`KDBoxWidget`)

Let's inspect `temperature` first using symmetrical dictionary inputs (`{label: array}`) so both axes are labeled automatically.

**Things to try in the widget below:**
1. **Heuristic vs. Mean-Shift Representative Lines:**
   * Notice that with the default **`Heuristic (Density + Slope)`**, step-to-step noise can cause multiple representative lines to cluster inside the dominant `Normal` bundle.
   * Switch **Rep Lines Method** to **`Mean-Shift (Curve Modes)`** (with `Rep Lines Count = 3`, `Diversity = 0.15`): it cleanly captures all three distinct operating regimes (*Normal*, *Thermal Surge*, and *Cooling Drift*).
2. **Brush (Timebox) & Strict Query Mode:**
   * Draw a rectangle around the high-temperature spike (`T ≈ 55--110 s`, `V ≈ 70--95 °C`). Notice that dragging edges/corners resizes the box and dragging the interior moves it.
   * Toggle **Strict query mode** on/off to see the difference between *Box Intersect* (any curve touching the box) and *Strict Timebox* (curves staying inside the box for the entire brushed time span).
3. **Angular Tool (`θ ± Δθ`):**
   * Switch **Tool** to **`Angular`** (notice **Strict query mode** hides automatically).
   * Click around `T ≈ 30 s, V ≈ 55 °C` and drag downward-right to match the `Cooling Drift` slope (roughly `-20°` to `-35°`). Drag the upper/lower corner handles of the shaded wedge to widen or narrow the `±deg°` tolerance.
4. **RNL & Pin Tools + Contrast:**
   * Try **`RNL (Radius)`** (resize by dragging the circle perimeter) and **`Pin (Hover)`** (3 px probe).
   * Drag **Contrast (Gamma)** below `1.0` (e.g., `0.5`) to brighten sparse trajectories, and toggle **Normalize density** to compare local vs. global intensity scaling.

In [5]:
w_temp = KDBoxWidget(
    {"Elapsed Time (s)": prep.time_data},
    {"Temperature (°C)": prep.value_data["temperature"]},
    attributes={"Plant": prep.attributes["plant"]},
    rep_method="meanshift",
    rep_count=3,
    diversity=0.15,
)
w_temp

## 3. Programmatic Control & Exporting Univariate Selections

Every UI parameter is two-way synced with Python traitlets, and any interactive selection you make above can be exported immediately in the next cell.

In [8]:
# Two-way programmatic control (updates the widget UI above live).
w_temp.colormap = "viridis"
w_temp.rep_method = "meanshift"
w_temp.rep_count = 3

# Export whatever is currently selected in the widget above.
sel_idx = w_temp.get_selected_indices()
sel_t, sel_v = w_temp.get_selected_data()
sel_attrs = w_temp.get_selected_attributes()
rep_idx = w_temp.get_representative_indices()

print(f"Selected runs      : {len(sel_idx)} / {w_temp.n_lines}")
print(f"Selected arrays    : time={sel_t.shape}, value={sel_v.shape}")
print(f"First 5 run IDs    : {prep.ids[sel_idx[:5]].tolist()}")
print(f"Representative IDs : {prep.ids[rep_idx].tolist()}")
print("Selected plant counts:", pd.Series(sel_attrs["Plant"]).value_counts().to_dict())

Selected runs      : 848 / 848
Selected arrays    : time=(848, 100), value=(848, 100)
First 5 run IDs    : ['RUN-0001', 'RUN-0002', 'RUN-0003', 'RUN-0004', 'RUN-0005']
Representative IDs : ['RUN-0031', 'RUN-0819', 'RUN-0649']
Selected plant counts: {'Plant North': 315, 'Plant East': 209, 'Plant South': 163, 'Plant West': 161}


## 4. Coordinated Multivariate Exploration (`MultivariateKDBox`)

Now let's link all three sensors --- **Temperature**, **Pressure**, and **Valve Opening** --- inside `MultivariateKDBox`.

Notice two layout optimizations:
* **Deduplicated Attribute Sidebar:** Only the first widget displays the **Attribute Filters** panel (though every change syncs across all three views). Because `Unit ID` has 30 unique values (`> 12`), a **live search box** appears above its dropdown automatically.
* **Coordinated Header Overlay:** The Matplotlib chart at the top plots each variable's representative lines on color-coded twin Y-axes (alternating left/right outward spines for 3+ variables).

**Guided scenario --- Root-Cause Investigation of the Thermal Surge:**
1. In **Temperature (°C)**, draw a **Brush** around the mid-run temperature spike (`T ≈ 55--110 s`, `V ≈ 70--95 °C`).
2. Look at **Pressure (bar)** and **Valve Opening (%)** below: both automatically switch to ***Coordinated Filter*** mode, immediately revealing that the thermal surge coincides with a sharp **pressure drop to ~2.6 bar** and the **valve snapping open to 100%**.
3. Look at the **top Matplotlib overlay**: a **blue shaded vertical band** marks the exact time window you brushed in `Temperature`, making it easy to verify whether the pressure drop and valve opening occur *during* or *after* the temperature spike.
4. Try filtering by **`Plant = Plant North`** or searching **`UNIT-001`** in the Attribute Filters sidebar, then run the final cell below to export the root-cause breakdown!

In [7]:
# Pass Plant and Unit ID as filterable attributes.
multi = MultivariateKDBox(
    {"Elapsed Time (s)": prep.time_data},
    {
        "Temperature (°C)": prep.value_data["temperature"],
        "Pressure (bar)": prep.value_data["pressure"],
        "Valve Opening (%)": prep.value_data["valve_pct"],
    },
    attributes={
        "Plant": prep.attributes["plant"],
        "Unit ID": prep.attributes["unit_id"],
    },
)

# Optionally switch all child widgets to Curve Mean-Shift for clean multi-modal summaries.
for child in multi.kd_widgets.values():
    child.rep_method = "meanshift"
    child.diversity = 0.15

multi

MultivariateKDBox(children=(Output(layout=Layout(display='flex', justify_content='center')), <kdbox.widget.KDB…

## 5. Top-Level Multivariate Export & Downstream Analysis

After brushing or filtering in `multi` above, pull the exact coordinated entity intersection back into Python with a single call and slice your original raw DataFrame.

In [8]:
# Extract coordinated selection across *all* views.
sel_idx = multi.get_selected_indices()
sel_run_ids = prep.ids[sel_idx]
sel_t, sel_values = multi.get_selected_data()
sel_attrs = multi.get_selected_attributes()

print(f"Coordinated selection: {len(sel_idx)} runs")
for var_name, arr in sel_values.items():
    if len(arr) > 0:
        print(f"  {var_name:<18s} -> shape={arr.shape}, min={arr.min():.2f}, max={arr.max():.2f}")

# Inspect categorical metadata & ground-truth regime composition of the brushed cluster.
summary_df = pd.DataFrame({
    "run_id": sel_run_ids,
    "plant": sel_attrs["Plant"],
    "unit_id": sel_attrs["Unit ID"],
    "true_regime": prep.attributes["true_regime"][sel_idx],
})

print("\nRegime breakdown of current selection:")
print(summary_df["true_regime"].value_counts())

print("\nTop units in current selection:")
print(summary_df["unit_id"].value_counts().head(8))

# Map back to the original unresampled raw DataFrame for downstream work.
raw_subset = df_raw[df_raw["run_id"].isin(sel_run_ids)]
print(f"\nMatched rows in original df_raw: {len(raw_subset):,} rows across {raw_subset['run_id'].nunique()} runs")

Coordinated selection: 848 runs
  Temperature (°C)   -> shape=(848, 100), min=27.84, max=93.15
  Pressure (bar)     -> shape=(848, 100), min=2.29, max=6.36
  Valve Opening (%)  -> shape=(848, 100), min=20.00, max=100.00

Regime breakdown of current selection:
true_regime
Normal           450
Thermal Surge    200
Cooling Drift    198
Name: count, dtype: int64

Top units in current selection:
unit_id
UNIT-006    62
UNIT-002    59
UNIT-003    56
UNIT-001    53
UNIT-005    52
UNIT-004    49
UNIT-007    22
UNIT-008    22
Name: count, dtype: int64

Matched rows in original df_raw: 46,118 rows across 848 runs
